# FIAP · Fase 6 · PBL "O despertar da Rede Neural"

**Aluno:** Willian Kauê Tobias do Carmo · **RM:** 570038 · **Grupo 64** · Tecnologia em Inteligência Artificial
**Cliente fictício:** FarmTech Solutions · **Caso de uso:** controle de acesso com verificação de EPI

## Contexto

A FarmTech Solutions expandiu seus serviços de IA para segurança patrimonial e controle de acesso de funcionários. Este notebook demonstra, para um cliente, um sistema de visão computacional que detecta dois objetos relevantes na portaria de uma área operacional:

- **capacete**: EPI obrigatório para entrar na área;
- **celular**: item restrito em determinadas áreas.

A escolha do par tem um motivo técnico além do narrativo: *celular* existe no dataset COCO (classe `cell phone`) e *capacete* não existe. Isso permite, na Entrega 2, medir o que uma YOLO padrão (pré-treinada no COCO) faz com cada classe e mostrar em que situação a customização da rede é indispensável.

## Como executar

1. Runtime → Change runtime type → **T4 GPU**.
2. **Com Google Drive** (`USAR_DRIVE = True`): o notebook monta o Drive e usa a pasta `MyDrive/fiap_fase6/`, onde ficam o dataset, os rótulos do Make Sense e os resultados dos treinos.
3. **Sem Google Drive** (ex.: correção): o notebook obtém o dataset e os rótulos do próprio repositório do GitHub, sem nenhuma configuração.
4. Execute as células em ordem (Runtime → Run all). Os dois treinos levam poucos minutos em GPU.

## Sumário

1. Setup · 2. Dataset · 3. Rotulação no Make Sense · 4. Treinamento YOLOv5 (30 vs 60 épocas) · 5. Validação e teste · 6. Inferência nas imagens de teste · 7. Conclusões da Entrega 1 · 8. Entrega 2: YOLO padrão e CNN do zero · 9. Comparação crítica das três abordagens · 10. Conclusão geral · Apêndice: reprodutibilidade do dataset · Referências

## 1. Setup

Todos os parâmetros do projeto ficam na célula abaixo. O restante do notebook deriva deles, então trocar o par de objetos, o número de épocas ou o tamanho de imagem é uma edição em um único lugar.

In [ ]:
# ============================================================
# PARÂMETROS DO PROJETO (único lugar que precisa ser editado)
# ============================================================
ALUNO = "Willian Kauê Tobias do Carmo"
RM = "570038"
GRUPO = "Grupo 64"
REPO_GITHUB = "https://github.com/willktdc/grupo64-fiap-fase6"   # <- confirmar o nome do repositório

# Classes do detector: nome interno (usado nos rótulos) -> nome da classe no Open Images.
# A ORDEM define o índice da classe (0 = celular, 1 = capacete). Use a mesma ordem ao criar as labels no Make Sense.
CLASSES = {"celular": "Mobile phone", "capacete": "Helmet"}
NOMES = list(CLASSES)

IMG = 640            # resolução de treino/inferência da YOLO
BATCH = 16
SEED = 0             # mesma semente nos dois treinos, para a comparação ser justa
EPOCAS = [30, 60]    # as duas simulações exigidas pelo enunciado (valores bem diferentes entre si)
CONF = 0.25          # confiança mínima para exibir uma detecção

USAR_DRIVE = True    # False para rodar sem Google Drive (o notebook baixa tudo do GitHub)

In [ ]:
# Bibliotecas de uso geral. As específicas (YOLOv5, TensorFlow) são carregadas nas seções em que entram.
import os, sys, re, time, json, shutil, zipfile, subprocess, random, glob
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from PIL import Image, ImageDraw

random.seed(SEED); np.random.seed(SEED)
pd.set_option("display.width", 160); pd.set_option("display.max_colwidth", 60)

# Confirma a GPU: sem ela o treino funciona, mas leva muito mais tempo.
!nvidia-smi -L || echo "ATENÇÃO: sem GPU. Vá em Runtime > Change runtime type > T4 GPU"

In [ ]:
# Pasta base: Google Drive (persistente entre sessões) ou pasta local (correção / execução sem Drive).
BASE = None
if USAR_DRIVE:
    try:
        from google.colab import drive
        drive.mount("/content/drive")
        BASE = Path("/content/drive/MyDrive/fiap_fase6")
    except Exception as e:                      # fora do Colab ou autorização negada
        print("Drive indisponível, usando pasta local. Motivo:", e)
if BASE is None:
    BASE = Path("/content/fiap_fase6")
BASE.mkdir(parents=True, exist_ok=True)

DATASET = BASE / "dataset"          # imagens, rótulos, manifesto, créditos
RUNS = BASE / "runs"                # resultados dos treinos (equivalente ao yolov5/runs, mas persistido)
RESULTADOS = BASE / "results"       # figuras e prints usados no README
LOCAL = Path("/content/dataset")    # cópia local do dataset (I/O do Drive é lento para treinar)
for p in (RUNS, RESULTADOS): p.mkdir(parents=True, exist_ok=True)
print("Pasta base:", BASE)

## 2. Dataset

### Fonte e critérios de seleção

As 80 imagens são um **subconjunto curado do [Open Images V7](https://storage.googleapis.com/openimages/web/index.html)** (Google), um dataset de fotos reais do Flickr publicadas sob licença Creative Commons (todas as imagens usadas aqui são CC BY 2.0; autor e link de cada uma estão em `dataset/CREDITOS.md`). A fonte foi escolhida por três razões:

1. **Realismo**: são fotos de pessoas comuns, com variação de ambiente, luz, ângulo e aparelho, bem diferente de fotos de estúdio ou de um único ambiente doméstico;
2. **Reprodutibilidade**: qualquer pessoa consegue recriar exatamente o mesmo dataset a partir dos IDs listados em `dataset/manifesto.csv` (ver Apêndice);
3. **Escala**: o filtro abaixo foi aplicado sobre ~1,7 mil imagens candidatas das duas classes, e as 80 finais foram escolhidas por inspeção visual.

Filtro automático aplicado às anotações oficiais (splits `validation` e `test` do Open Images):

| Critério | Motivo |
|---|---|
| Exatamente **uma** caixa da classe na imagem | uma classe por foto, exigência da CNN de classificação da Entrega 2 |
| Caixa ocupando **≥ 12%** da imagem | objeto visível, como numa foto apontada para ele |
| Sem `IsGroupOf`, `IsDepiction`, `IsInside` | nada de multidões, desenhos ou interiores |
| Imagem **não contém a outra classe** (nem parentes: *Bicycle helmet*, *Football helmet*, *Telephone*) | evita ambiguidade nos rótulos |
| Rotação EXIF conhecida e igual a 0 | para as caixas oficiais baterem com os pixels |

Na inspeção visual foram descartadas réplicas de museu, máscaras, adereços de fantasia, renders de produto, telefones sem fio e brinquedos, deixando um conjunto com **mistura deliberada de escalas**: metade com o objeto em close e metade com o objeto em contexto (na cabeça de um operário, na mão de uma pessoa).

### Organização

- Split **estratificado por classe**, com semente fixa: 32 treino / 4 validação / 4 teste para cada classe (64 / 8 / 8 no total), conforme o enunciado;
- Imagens redimensionadas para no máximo 1024 px no maior lado, sem distorção;
- Estrutura no Google Drive (`MyDrive/fiap_fase6/dataset/`), no padrão esperado pela YOLOv5:

```
dataset/
├── images/{train,val,test}/        celular_001.jpg … capacete_040.jpg
├── labels/{train,val,test}/        rótulos criados no Make Sense (seção 3)
├── gabarito_openimages/{...}/      caixas oficiais do Open Images, usadas só para conferir a rotulação
├── manifesto.csv                   arquivo, classe, split, ID no Open Images, caixa oficial, autor, licença
└── CREDITOS.md                     atribuição imagem a imagem
```

In [ ]:
# Obtém o dataset, nesta ordem: (1) já está na pasta base; (2) zip na pasta base; (3) clone do repositório do GitHub.
def dataset_ok(p):
    return len(list((p / "images" / "train").glob("*.jpg"))) == 64 and (p / "manifesto.csv").exists()

if not dataset_ok(DATASET):
    z = BASE / "dataset_fase6.zip"
    if z.exists():
        print("Extraindo", z.name)
        with zipfile.ZipFile(z) as zf:
            zf.extractall(DATASET)
    else:
        print("Baixando o dataset do repositório:", REPO_GITHUB)
        tmp = Path("/content/repo_tmp"); shutil.rmtree(tmp, ignore_errors=True)
        subprocess.run(["git", "clone", "--depth", "1", REPO_GITHUB, str(tmp)], check=True)
        shutil.copytree(tmp / "dataset", DATASET, dirs_exist_ok=True)
assert dataset_ok(DATASET), "Dataset não encontrado: coloque dataset_fase6.zip em " + str(BASE)

manifesto = pd.read_csv(DATASET / "manifesto.csv")
print("Imagens por classe e split:")
display(manifesto.groupby(["classe", "split"]).size().unstack())
print("Licenças:", manifesto.licenca.value_counts().to_dict())

In [ ]:
# Amostra do dataset: 8 imagens de treino de cada classe, para dar uma ideia da variedade.
def mostrar_grade(caminhos, titulos=None, cols=4, tam=3.4, titulo=None, salvar=None):
    """Exibe uma grade de imagens (e opcionalmente salva a figura em RESULTADOS)."""
    caminhos = [Path(c) for c in caminhos]
    titulos = titulos or [c.name for c in caminhos]
    linhas = (len(caminhos) + cols - 1) // cols
    fig, eixos = plt.subplots(linhas, cols, figsize=(cols * tam, linhas * tam * 0.85))
    eixos = np.array(eixos).ravel()
    for eixo, c, t in zip(eixos, caminhos, titulos):
        eixo.imshow(Image.open(c)); eixo.set_title(t, fontsize=9); eixo.axis("off")
    for eixo in eixos[len(caminhos):]:
        eixo.axis("off")
    if titulo: fig.suptitle(titulo, fontsize=12)
    plt.tight_layout()
    if salvar: fig.savefig(RESULTADOS / salvar, dpi=110, bbox_inches="tight")
    plt.show()

amostra = [DATASET / "images/train" / a for a in manifesto[manifesto.split == "train"].groupby("classe").head(8).arquivo]
mostrar_grade(amostra, cols=8, tam=2.4, titulo="Amostra do dataset (treino)", salvar="amostra_dataset.png")

## 3. Rotulação no Make Sense

A rotulação das caixas foi feita manualmente no [Make Sense](https://www.makesense.ai), conforme o enunciado, para **todas as 80 imagens** (e não só as de treino): a YOLOv5 precisa dos rótulos de validação para calcular mAP a cada época, e os de teste permitem medir o desempenho final no conjunto que o modelo nunca viu.

Passo a passo executado:

1. `makesense.ai` → *Get Started* → arrastar as 80 imagens (pastas `images/train`, `val` e `test`) → *Object Detection*;
2. Criar as labels **nesta ordem**: `celular` e depois `capacete` (a ordem vira o índice da classe no formato YOLO e precisa bater com `NOMES`);
3. Em cada imagem, desenhar **uma** caixa (*Rect*) ao redor do objeto e atribuir a label;
4. *Actions → Export Annotations → A .zip package containing files in YOLO format*;
5. Salvar o zip em `MyDrive/fiap_fase6/makesense/` (qualquer nome terminado em `.zip`; se a rotulação for feita em etapas, cada exportação gera um zip e todos são lidos).

Formato YOLO gerado pelo Make Sense: um `.txt` por imagem, com uma linha por caixa `classe x_centro y_centro largura altura`, tudo normalizado entre 0 e 1.

A célula seguinte distribui os `.txt` exportados nas pastas `labels/{train,val,test}` de acordo com o split de cada imagem. Se os rótulos já existirem (caso da correção, em que vêm do repositório), ela apenas confirma a contagem.

In [ ]:
# Distribui os rótulos exportados do Make Sense nas pastas labels/{train,val,test}.
PASTA_MS = BASE / "makesense"; PASTA_MS.mkdir(exist_ok=True)
split_de = dict(zip(manifesto.arquivo.str.replace(".jpg", "", regex=False), manifesto.split))
for sp in ["train", "val", "test"]:
    (DATASET / "labels" / sp).mkdir(parents=True, exist_ok=True)

zips = sorted(PASTA_MS.glob("*.zip"))
n = 0
for z in zips:
    with zipfile.ZipFile(z) as zf:
        for info in zf.infolist():
            nome = Path(info.filename).stem
            if info.filename.endswith(".txt") and nome in split_de:
                (DATASET / "labels" / split_de[nome] / (nome + ".txt")).write_bytes(zf.read(info)); n += 1
print(f"zips lidos: {[z.name for z in zips]} | rótulos distribuídos agora: {n}")

contagem = {sp: len(list((DATASET / "labels" / sp).glob("*.txt"))) for sp in ["train", "val", "test"]}
print("rótulos existentes por split:", contagem)
assert contagem == {"train": 64, "val": 8, "test": 8}, "Faltam rótulos: exporte do Make Sense e rode de novo"

In [ ]:
# Verificação dos rótulos: uma caixa por imagem, classe coerente com o nome do arquivo, e visualização.
def ler_caixas(txt):
    """Lê um .txt YOLO e devolve [(classe, cx, cy, w, h), ...]."""
    linhas = [l.split() for l in Path(txt).read_text().strip().splitlines() if l.strip()]
    return [(int(l[0]), *map(float, l[1:5])) for l in linhas]

problemas, registros = [], []
for sp in ["train", "val", "test"]:
    for img in sorted((DATASET / "images" / sp).glob("*.jpg")):
        caixas = ler_caixas(DATASET / "labels" / sp / (img.stem + ".txt"))
        classe_nome = img.stem.split("_")[0]
        if len(caixas) != 1: problemas.append(f"{img.name}: {len(caixas)} caixas")
        elif NOMES[caixas[0][0]] != classe_nome: problemas.append(f"{img.name}: rótulo {NOMES[caixas[0][0]]} ≠ {classe_nome}")
        registros.append(dict(split=sp, classe=classe_nome, n_caixas=len(caixas), area=caixas[0][3] * caixas[0][4] if caixas else 0))
print("problemas encontrados:", problemas or "nenhum")
display(pd.DataFrame(registros).groupby(["split", "classe"]).agg(imagens=("n_caixas", "size"), caixas=("n_caixas", "sum"), area_media=("area", "mean")).round(2))

def desenhar(img_path, txt_path, cor="lime", largura=4, im=None):
    """Desenha as caixas YOLO de txt_path sobre a imagem (devolve PIL.Image)."""
    im = im or Image.open(img_path).convert("RGB"); W, H = im.size; d = ImageDraw.Draw(im)
    for c, cx, cy, w, h in ler_caixas(txt_path):
        d.rectangle([(cx - w / 2) * W, (cy - h / 2) * H, (cx + w / 2) * W, (cy + h / 2) * H], outline=cor, width=largura)
        d.text(((cx - w / 2) * W + 6, (cy - h / 2) * H + 6), NOMES[c], fill=cor)
    return im

exemplos = [DATASET / "images/train" / a for a in manifesto[manifesto.split == "train"].groupby("classe").head(4).arquivo]
fig, eixos = plt.subplots(2, 4, figsize=(14, 7))
for eixo, p in zip(eixos.ravel(), exemplos):
    eixo.imshow(desenhar(p, DATASET / "labels/train" / (p.stem + ".txt"))); eixo.set_title(p.name, fontsize=9); eixo.axis("off")
fig.suptitle("Rótulos criados no Make Sense (verde)"); plt.tight_layout(); fig.savefig(RESULTADOS / "rotulos_makesense.png", dpi=110, bbox_inches="tight"); plt.show()

### Conferência da rotulação contra o gabarito do Open Images

O Open Images traz uma caixa anotada por profissionais para cada uma dessas imagens. Ela **não foi usada para treinar**; serve como controle de qualidade da rotulação manual: o IoU (interseção sobre união) entre a minha caixa e a oficial mostra se a rotulação foi consistente. IoU acima de 0,7 é o patamar usado pelo próprio Open Images para considerar uma caixa correta.

In [ ]:
# IoU entre a caixa do Make Sense e a caixa oficial do Open Images, imagem a imagem.
def iou(a, b):
    """IoU de duas caixas no formato YOLO (cx, cy, w, h)."""
    ax1, ay1, ax2, ay2 = a[0] - a[2] / 2, a[1] - a[3] / 2, a[0] + a[2] / 2, a[1] + a[3] / 2
    bx1, by1, bx2, by2 = b[0] - b[2] / 2, b[1] - b[3] / 2, b[0] + b[2] / 2, b[1] + b[3] / 2
    inter = max(0, min(ax2, bx2) - max(ax1, bx1)) * max(0, min(ay2, by2) - max(ay1, by1))
    uniao = a[2] * a[3] + b[2] * b[3] - inter
    return inter / uniao if uniao else 0

ious = []
for sp in ["train", "val", "test"]:
    for txt in sorted((DATASET / "labels" / sp).glob("*.txt")):
        minha = ler_caixas(txt); oficial = ler_caixas(DATASET / "gabarito_openimages" / sp / txt.name)
        if minha and oficial:
            ious.append(dict(imagem=txt.stem, split=sp, iou=iou(minha[0][1:], oficial[0][1:])))
df_iou = pd.DataFrame(ious)
print(f"IoU médio: {df_iou.iou.mean():.3f} | mediana: {df_iou.iou.median():.3f} | mínimo: {df_iou.iou.min():.3f} | imagens com IoU < 0,7: {(df_iou.iou < 0.7).sum()}")
display(df_iou.nsmallest(5, "iou").round(3))

# As 3 piores concordâncias, para inspeção: verde = Make Sense, vermelho = Open Images
piores = df_iou.nsmallest(3, "iou")
fig, eixos = plt.subplots(1, 3, figsize=(15, 5))
for eixo, r in zip(eixos, piores.itertuples()):
    p = DATASET / "images" / r.split / (r.imagem + ".jpg")
    im = desenhar(p, DATASET / "labels" / r.split / (r.imagem + ".txt"), "lime")
    im = desenhar(p, DATASET / "gabarito_openimages" / r.split / (r.imagem + ".txt"), "red", im=im)
    eixo.imshow(im); eixo.set_title(f"{r.imagem}  IoU={r.iou:.2f}", fontsize=10); eixo.axis("off")
plt.tight_layout(); plt.show()

In [ ]:
# Cópia local do dataset (treino mais rápido que lendo do Drive) e arquivo data.yaml da YOLOv5.
shutil.rmtree(LOCAL, ignore_errors=True)
shutil.copytree(DATASET / "images", LOCAL / "images")
shutil.copytree(DATASET / "labels", LOCAL / "labels")

DATA_YAML = LOCAL / "data.yaml"
DATA_YAML.write_text(f"path: {LOCAL}\ntrain: images/train\nval: images/val\ntest: images/test\nnc: {len(NOMES)}\nnames: {NOMES}\n")
print(DATA_YAML.read_text())

## 4. Treinamento com YOLOv5

### Abordagem

Usei a implementação oficial da [YOLOv5 (Ultralytics)](https://github.com/ultralytics/yolov5), a mesma do material da disciplina, partindo dos pesos `yolov5s.pt` pré-treinados no COCO. Ou seja, a "customização" é um **fine-tuning**: a rede já sabe extrair características visuais genéricas e aprende apenas a localizar e classificar as duas classes novas. É por isso que 64 imagens bastam para um resultado útil, algo impossível treinando um detector do zero.

Parâmetros fixos nas duas simulações, para que a única variável seja o número de épocas:

| Parâmetro | Valor | Observação |
|---|---|---|
| Modelo | `yolov5s` | menor da família, suficiente para 2 classes e rápido em GPU T4 |
| Resolução | 640 px | padrão da YOLOv5 |
| Batch | 16 | |
| Semente | 0 | mesma inicialização e mesma ordem de dados nos dois treinos |
| Épocas | **30** e **60** | valores bem diferentes, conforme o enunciado |
| Augmentation | padrão da YOLOv5 (mosaic, HSV, flip, escala) | compensa o dataset pequeno |

Cada treino grava em `runs/train/expN/` (aqui persistido no Drive): `best.pt` e `last.pt`, `results.csv` com as métricas por época, `results.png`, matriz de confusão, curvas PR e F1, e lotes de validação com as predições desenhadas.

In [ ]:
# Instala a YOLOv5 (clone do repositório oficial + dependências).
YOLO = Path("/content/yolov5")
if not YOLO.exists():
    !git clone -q https://github.com/ultralytics/yolov5 /content/yolov5
%cd /content/yolov5
!pip install -qr requirements.txt

import torch
print("torch", torch.__version__, "| GPU disponível:", torch.cuda.is_available())

In [ ]:
TEMPOS, METRICAS = {}, {}     # tempos de treino (s) e métricas de cada abordagem, usados na comparação final

def run_cmd(cmd):
    """Executa um comando de terminal mostrando a saída em tempo real. Devolve (duração em s, saída completa)."""
    print(">>", cmd, "\n")
    t0 = time.time(); linhas = []
    p = subprocess.Popen(cmd, shell=True, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, bufsize=1)
    for linha in p.stdout:
        print(linha, end=""); linhas.append(linha)
    p.wait()
    assert p.returncode == 0, f"comando falhou com código {p.returncode}"
    return time.time() - t0, "".join(linhas)

def treinar(epocas):
    """Treina a YOLOv5s por `epocas` épocas e devolve a pasta do experimento."""
    nome = f"exp{epocas}"
    dur, _ = run_cmd(f"python train.py --img {IMG} --batch {BATCH} --epochs {epocas} --data {DATA_YAML} "
                     f"--weights yolov5s.pt --seed {SEED} --cache ram --project {RUNS}/train --name {nome} --exist-ok")
    TEMPOS[nome] = dur
    print(f"\n⏱ Treino {nome}: {dur / 60:.1f} min")
    return RUNS / "train" / nome

### Simulação 1: 30 épocas

In [ ]:
EXP = {}
EXP[30] = treinar(30)

### Simulação 2: 60 épocas

In [ ]:
EXP[60] = treinar(60)

## 5. Validação e teste

### Métricas usadas

- **Precision (P)**: das caixas que o modelo previu, quantas estavam certas;
- **Recall (R)**: dos objetos que existiam, quantos o modelo encontrou;
- **mAP@0.5**: precisão média considerando correta uma caixa com IoU ≥ 0,5 com o gabarito; é a métrica mais usada para detecção;
- **mAP@0.5:0.95**: o mesmo, mas com a exigência de IoU variando de 0,5 a 0,95; mede a *qualidade do enquadramento*, não só se o objeto foi achado;
- **Losses** (`box`, `obj`, `cls`): erro de posição da caixa, de "existe objeto aqui" e de classe. A comparação entre a loss de treino e a de validação é o termômetro de overfitting.

As curvas por época vêm do `results.csv` de cada treino. As métricas finais de validação e de **teste** (8 imagens que o modelo nunca viu) são calculadas com o `val.py` da YOLOv5 usando o `best.pt` de cada simulação.

In [ ]:
# Curvas de treino: mAP@0.5 e losses por época, 30 vs 60 épocas sobrepostos.
def ler_resultados(pasta):
    df = pd.read_csv(pasta / "results.csv"); df.columns = [c.strip() for c in df.columns]; return df

res = {e: ler_resultados(p) for e, p in EXP.items()}
fig, eixos = plt.subplots(1, 3, figsize=(17, 4.5))
for e, df in res.items():
    eixos[0].plot(df["epoch"], df["metrics/mAP_0.5"], label=f"{e} épocas")
    eixos[1].plot(df["epoch"], df["train/box_loss"] + df["train/obj_loss"] + df["train/cls_loss"], label=f"treino {e}")
    eixos[1].plot(df["epoch"], df["val/box_loss"] + df["val/obj_loss"] + df["val/cls_loss"], "--", label=f"validação {e}")
    eixos[2].plot(df["epoch"], df["metrics/mAP_0.5:0.95"], label=f"{e} épocas")
for eixo, t in zip(eixos, ["mAP@0.5 (validação)", "Loss total (treino sólido, validação tracejado)", "mAP@0.5:0.95 (validação)"]):
    eixo.set_title(t); eixo.set_xlabel("época"); eixo.grid(alpha=.3); eixo.legend()
plt.tight_layout(); fig.savefig(RESULTADOS / "curvas_30_vs_60.png", dpi=110, bbox_inches="tight"); plt.show()

# Painel completo que a própria YOLOv5 gera (results.png) para cada simulação
for e, p in EXP.items():
    display(Image.open(p / "results.png").resize((1400, 700)))
    print(f"↑ results.png da simulação com {e} épocas\n")

In [ ]:
# Métricas finais em VALIDAÇÃO e em TESTE com o best.pt de cada simulação (API do val.py da YOLOv5).
sys.path.insert(0, str(YOLO))
import val as yolo_val

linhas = []
for e, p in EXP.items():
    for tarefa in ["val", "test"]:
        (mp, mr, map50, map5095, *perdas), _, tempos_ms = yolo_val.run(
            data=str(DATA_YAML), weights=str(p / "weights/best.pt"), imgsz=IMG, task=tarefa,
            project=str(RUNS / "val"), name=f"exp{e}_{tarefa}", exist_ok=True, plots=True, verbose=True)
        linhas.append(dict(modelo=f"YOLOv5 custom {e} ép.", conjunto=tarefa, precision=mp, recall=mr,
                           mAP50=map50, **{"mAP50-95": map5095}, inferencia_ms=tempos_ms[1], treino_s=TEMPOS[f"exp{e}"]))
tabela_yolo = pd.DataFrame(linhas)
display(tabela_yolo.round(3))
tabela_yolo.to_csv(RESULTADOS / "metricas_yolo_custom.csv", index=False)

# Guarda o que entra na comparação final (Entrega 2)
for e in EXP:
    t = tabela_yolo[(tabela_yolo.modelo == f"YOLOv5 custom {e} ép.") & (tabela_yolo.conjunto == "test")].iloc[0]
    METRICAS[f"YOLOv5 custom {e} ép."] = dict(mAP50_teste=t.mAP50, inferencia_ms=t.inferencia_ms, treino_s=t.treino_s)

In [ ]:
# Matrizes de confusão (geradas pela YOLOv5 no fim de cada treino) lado a lado.
fig, eixos = plt.subplots(1, 2, figsize=(14, 6))
for eixo, (e, p) in zip(eixos, EXP.items()):
    eixo.imshow(Image.open(p / "confusion_matrix.png")); eixo.set_title(f"{e} épocas"); eixo.axis("off")
plt.tight_layout(); fig.savefig(RESULTADOS / "matrizes_confusao.png", dpi=110, bbox_inches="tight"); plt.show()

> ✍️ **[PREENCHER após os treinos] Leitura das curvas e das métricas (30 vs 60 épocas)**
>
> - Com 30 épocas o mAP@0.5 de validação chegou a ___ e com 60 a ___; no teste, ___ e ___. O ganho de dobrar as épocas foi de ___ pontos, ao custo de ___ min a mais de treino.
> - A loss de validação (tracejada) ___ [continuou caindo / estabilizou / subiu] a partir da época ___, o que indica ___ [que ainda havia o que aprender / saturação / início de overfitting].
> - mAP@0.5:0.95 ___ [subiu mais / menos] que o mAP@0.5, ou seja, o treino mais longo melhorou principalmente ___ [o enquadramento das caixas / a capacidade de achar o objeto].
> - Na matriz de confusão, a confusão mais comum foi ___ (ex.: capacete previsto como fundo), que faz sentido porque ___.

## 6. Inferência nas imagens de teste (prints para o cliente)

Aqui o modelo é usado como seria na operação: recebe as 8 fotos de teste, que nunca viu, e devolve as caixas com a classe e a confiança. O `detect.py` grava as imagens anotadas em `runs/detect/expN/` (persistido no Drive) e os rótulos previstos em `labels/`, que uso a seguir para medir o acerto por imagem.

In [ ]:
def detectar(pesos, nome, fonte=None, conf=CONF):
    """Roda o detect.py sobre as imagens de teste e devolve (pasta de saída, ms por imagem na inferência)."""
    fonte = fonte or (LOCAL / "images/test")
    dur, saida = run_cmd(f"python detect.py --weights {pesos} --source {fonte} --img {IMG} --conf {conf} "
                         f"--project {RUNS}/detect --name {nome} --exist-ok --save-txt --save-conf --line-thickness 3")
    m = re.search(r"([\d.]+)ms pre-process, ([\d.]+)ms inference, ([\d.]+)ms NMS", saida)
    return RUNS / "detect" / nome, (float(m.group(2)) if m else float("nan"))

DET = {}
for e, p in EXP.items():
    DET[e] = detectar(p / "weights/best.pt", f"exp{e}_teste")

# Prints: as 8 imagens de teste processadas por cada modelo
for e, (pasta, ms) in DET.items():
    imagens = sorted(pasta.glob("*.jpg"))
    mostrar_grade(imagens, cols=4, tam=3.6, titulo=f"Detecções nas imagens de teste: YOLOv5 custom, {e} épocas ({ms:.1f} ms/imagem)",
                  salvar=f"testes_detectados_{e}ep.png")

In [ ]:
# Acerto por imagem: a classe verdadeira (prefixo do nome do arquivo) aparece entre as detecções?
# Esta métrica é mais simples que o mAP, mas tem uma vantagem: pode ser aplicada igualmente às três abordagens da Entrega 2
# (detector customizado, detector padrão do COCO e classificador CNN), que não compartilham a mesma métrica nativa.
def acerto_por_imagem(pasta_labels, mapa_indice_para_nome, modelo):
    linhas = []
    for img in sorted((LOCAL / "images/test").glob("*.jpg")):
        verdade = img.stem.split("_")[0]
        txt = Path(pasta_labels) / (img.stem + ".txt")
        previstas = {}
        if txt.exists():
            for l in txt.read_text().splitlines():
                partes = l.split(); nome = mapa_indice_para_nome.get(int(partes[0]), f"classe {partes[0]}")
                previstas[nome] = max(previstas.get(nome, 0), float(partes[5]) if len(partes) > 5 else 0)
        linhas.append(dict(modelo=modelo, imagem=img.name, verdade=verdade,
                           previstas=", ".join(f"{k} ({v:.2f})" for k, v in sorted(previstas.items(), key=lambda x: -x[1])) or "(nenhuma)",
                           acertou=verdade in previstas,
                           falso_positivo=any(k in NOMES and k != verdade for k in previstas)))
    return pd.DataFrame(linhas)

ACERTOS = {}
mapa_custom = dict(enumerate(NOMES))
for e, (pasta, ms) in DET.items():
    ACERTOS[f"YOLOv5 custom {e} ép."] = acerto_por_imagem(pasta / "labels", mapa_custom, f"YOLOv5 custom {e} ép.")
    METRICAS[f"YOLOv5 custom {e} ép."]["acerto_imagem"] = ACERTOS[f"YOLOv5 custom {e} ép."].acertou.mean()
    METRICAS[f"YOLOv5 custom {e} ép."]["inferencia_ms_detect"] = ms
display(pd.concat(ACERTOS.values()).drop(columns="modelo").set_index("imagem"))
print({k: f"{v.acertou.mean():.0%} de acerto, {v.falso_positivo.sum()} falso(s) positivo(s)" for k, v in ACERTOS.items()})

## 7. Conclusões da Entrega 1

> ✍️ **[PREENCHER] Achados**
>
> - Resultado para o cliente: o detector customizado reconheceu ___ das 8 imagens de teste com ___ épocas e ___ com ___ épocas, com mAP@0.5 de teste de ___ e ___. Em termos práticos, ___ [ex.: o sistema identifica capacete e celular em fotos de celular comuns, com confiança média de ___].
> - Efeito das épocas: ___ [ex.: 60 épocas melhoraram o enquadramento (mAP@0.5:0.95) mas não a detecção em si; o custo foi ___ min a mais].
> - Erros observados: ___ [ex.: falha em capacete visto de lado / confiança baixa em celular escuro contra fundo escuro].

**Pontos fortes**

- Fine-tuning de um modelo pré-treinado: resultado útil com apenas 32 imagens de treino por classe e minutos de GPU;
- Dataset com variedade real de cenas, aparelhos e escalas, reproduzível a partir do `manifesto.csv`;
- Rotulação manual conferida contra um gabarito independente (IoU médio de ___);
- Experimento controlado: a única diferença entre as duas simulações é o número de épocas.

**Limitações**

- **Tamanho do dataset**: 4 imagens de validação e 4 de teste por classe tornam as métricas ruidosas; uma única imagem errada muda o mAP em mais de 10 pontos. Qualquer conclusão sobre "30 vs 60" precisa dessa ressalva;
- **Uma classe por imagem**: o modelo não foi exposto a cenas com celular e capacete juntos, exatamente o que aconteceria na portaria; o próximo passo seria incluir imagens mistas;
- **Domínio**: as fotos vêm do Flickr, não da câmera da portaria do cliente; iluminação, ângulo e resolução reais podem degradar o desempenho e exigiriam um ciclo de coleta no local;
- **Capacete é uma classe heterogênea** (obra, moto, esporte): útil para generalizar, mas o cliente pode querer distinguir o EPI específico da empresa.

## 8. Entrega 2: abordagens concorrentes

A partir do mesmo dataset, comparo três caminhos:

| Abordagem | O que é | Tarefa | Treino necessário |
|---|---|---|---|
| **YOLOv5 customizada** (Entrega 1) | `yolov5s` com fine-tuning nas 2 classes | detecção (caixa + classe) | sim, minutos |
| **YOLO padrão** (capítulo 3 de Redes Neurais) | `yolov5s.pt` pré-treinada no COCO, usada como veio, sem nenhum treino | detecção nas 80 classes do COCO | nenhum |
| **CNN do zero** | rede convolucional pequena, definida e treinada aqui, só com as 64 imagens de treino | classificação (a imagem inteira é "celular" ou "capacete") | sim, segundos |

Duas observações importantes para a leitura dos resultados:

1. A YOLO padrão **conhece `cell phone`** (classe 67 do COCO) mas **não conhece capacete**. A expectativa é que acerte celulares e, para capacetes, devolva outra coisa (`person`, provavelmente) ou nada. O resultado mostra o limite de usar um modelo pronto fora do seu vocabulário;
2. A CNN do zero resolve uma **tarefa diferente** (classificar, não localizar). Ela não diz *onde* está o objeto, só *qual* é. Por isso a comparação de precisão entre as três usa o **acerto por imagem**, a única métrica aplicável às três, além das métricas nativas de cada uma.

### 8.1 YOLO padrão (pesos COCO, sem treino)

In [ ]:
# Nomes das 80 classes do COCO, para traduzir os índices previstos pela YOLO padrão.
import yaml
coco = yaml.safe_load(open(YOLO / "data/coco128.yaml"))["names"]
coco = dict(enumerate(coco)) if isinstance(coco, list) else {int(k): v for k, v in coco.items()}
mapa_coco = dict(coco); mapa_coco[67] = "celular"      # 'cell phone' do COCO equivale à nossa classe celular
print("Classe do COCO que corresponde a celular:", coco[67], "| alguma classe 'helmet' no COCO?", any("helmet" in v for v in coco.values()))

pasta_coco, ms_coco = detectar(YOLO / "yolov5s.pt", "yolo_padrao_coco_teste")
mostrar_grade(sorted(pasta_coco.glob("*.jpg")), cols=4, tam=3.6,
              titulo=f"YOLO padrão (COCO, sem treino) nas imagens de teste ({ms_coco:.1f} ms/imagem)", salvar="testes_yolo_padrao.png")

ACERTOS["YOLO padrão (COCO)"] = acerto_por_imagem(pasta_coco / "labels", mapa_coco, "YOLO padrão (COCO)")
display(ACERTOS["YOLO padrão (COCO)"].drop(columns="modelo").set_index("imagem"))
METRICAS["YOLO padrão (COCO)"] = dict(mAP50_teste=np.nan, inferencia_ms_detect=ms_coco, treino_s=0,
                                     acerto_imagem=ACERTOS["YOLO padrão (COCO)"].acertou.mean())
print(f"Acerto por imagem: {METRICAS['YOLO padrão (COCO)']['acerto_imagem']:.0%}")

> ✍️ **[PREENCHER] YOLO padrão**: nas 4 fotos de celular ela ___ [detectou `cell phone` em ___ delas]; nas 4 de capacete devolveu ___ [`person` / nada], confirmando que ___. O mAP não é calculável aqui porque o espaço de classes da rede (80 do COCO) é diferente do nosso, por isso a comparação é pelo acerto por imagem.

### 8.2 CNN treinada do zero (classificação)

Arquitetura deliberadamente pequena, adequada a 64 imagens: três blocos *Conv → MaxPool*, achatamento, *Dropout* de 50% e uma camada densa. A entrada é reduzida a 160 × 160 px e recebe augmentation leve (flip, rotação, zoom) para compensar o dataset. A rede aprende tudo do zero: nenhum peso pré-treinado.

In [ ]:
# Organiza as imagens em pastas por classe (formato esperado por image_dataset_from_directory).
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers
tf.random.set_seed(SEED)

CLS_DIR = Path("/content/classificacao"); shutil.rmtree(CLS_DIR, ignore_errors=True)
for sp in ["train", "val", "test"]:
    for p in (LOCAL / "images" / sp).glob("*.jpg"):
        destino = CLS_DIR / sp / p.stem.split("_")[0]; destino.mkdir(parents=True, exist_ok=True); shutil.copy(p, destino / p.name)

IMG_CNN = 160
carregar = lambda sp, embaralhar: keras.utils.image_dataset_from_directory(
    CLS_DIR / sp, image_size=(IMG_CNN, IMG_CNN), batch_size=8, shuffle=embaralhar, seed=SEED)
ds_treino, ds_val, ds_teste = carregar("train", True), carregar("val", False), carregar("test", False)
classes_cnn = ds_treino.class_names          # ordem alfabética: ['capacete', 'celular']
arquivos_teste = [Path(p).name for p in ds_teste.file_paths]   # ordem das imagens de teste (shuffle=False)
print("classes da CNN:", classes_cnn)
AUTO = tf.data.AUTOTUNE
ds_treino, ds_val, ds_teste = (d.cache().prefetch(AUTO) for d in (ds_treino, ds_val, ds_teste))

cnn = keras.Sequential([
    layers.Input((IMG_CNN, IMG_CNN, 3)),
    layers.RandomFlip("horizontal"), layers.RandomRotation(0.05), layers.RandomZoom(0.1),   # augmentation (só no treino)
    layers.Rescaling(1.0 / 255),                                                             # normaliza para [0, 1]
    layers.Conv2D(32, 3, activation="relu"), layers.MaxPooling2D(),                          # bloco 1: bordas e texturas
    layers.Conv2D(64, 3, activation="relu"), layers.MaxPooling2D(),                          # bloco 2: partes
    layers.Conv2D(128, 3, activation="relu"), layers.MaxPooling2D(),                         # bloco 3: formas
    layers.Flatten(), layers.Dropout(0.5), layers.Dense(128, activation="relu"),             # classificador
    layers.Dense(len(classes_cnn), activation="softmax"),
], name="cnn_do_zero")
cnn.compile(optimizer="adam", loss="sparse_categorical_crossentropy", metrics=["accuracy"])
cnn.summary()

t0 = time.time()
historico = cnn.fit(ds_treino, validation_data=ds_val, epochs=25, verbose=2)
TEMPOS["cnn"] = time.time() - t0
print(f"\n⏱ Treino da CNN: {TEMPOS['cnn']:.0f} s")

In [ ]:
# Avaliação da CNN: curvas, acurácia em validação e teste, matriz de confusão e tempo de inferência por imagem.
from sklearn.metrics import confusion_matrix, classification_report

h = historico.history
fig, eixos = plt.subplots(1, 2, figsize=(12, 4))
eixos[0].plot(h["accuracy"], label="treino"); eixos[0].plot(h["val_accuracy"], label="validação"); eixos[0].set_title("Acurácia"); eixos[0].legend(); eixos[0].grid(alpha=.3)
eixos[1].plot(h["loss"], label="treino"); eixos[1].plot(h["val_loss"], label="validação"); eixos[1].set_title("Loss"); eixos[1].legend(); eixos[1].grid(alpha=.3)
plt.tight_layout(); fig.savefig(RESULTADOS / "cnn_curvas.png", dpi=110, bbox_inches="tight"); plt.show()

perda_val, acc_val = cnn.evaluate(ds_val, verbose=0)
perda_teste, acc_teste = cnn.evaluate(ds_teste, verbose=0)
print(f"acurácia validação: {acc_val:.2%} | acurácia teste: {acc_teste:.2%}")

y_verd = np.concatenate([y.numpy() for _, y in ds_teste]); y_prev = np.argmax(cnn.predict(ds_teste, verbose=0), axis=1)
print(classification_report(y_verd, y_prev, target_names=classes_cnn, zero_division=0))
display(pd.DataFrame(confusion_matrix(y_verd, y_prev), index=[f"real {c}" for c in classes_cnn], columns=[f"prev. {c}" for c in classes_cnn]))

# Tempo de inferência por imagem (média de 8 imagens, após 1 chamada de aquecimento)
lote = next(iter(ds_teste))[0]; cnn.predict(lote[:1], verbose=0)
t0 = time.time()
for i in range(len(lote)): cnn.predict(lote[i:i + 1], verbose=0)
ms_cnn = (time.time() - t0) / len(lote) * 1000

# Acerto por imagem no mesmo formato das outras abordagens
ACERTOS["CNN do zero"] = pd.DataFrame(dict(modelo="CNN do zero", imagem=arquivos_teste, verdade=[classes_cnn[i] for i in y_verd],
                                           previstas=[classes_cnn[i] for i in y_prev], acertou=(y_verd == y_prev), falso_positivo=(y_verd != y_prev)))
display(ACERTOS["CNN do zero"].drop(columns="modelo").set_index("imagem"))
METRICAS["CNN do zero"] = dict(mAP50_teste=np.nan, inferencia_ms_detect=ms_cnn, treino_s=TEMPOS["cnn"], acerto_imagem=float(acc_teste))

> ✍️ **[PREENCHER] CNN do zero**: acurácia de ___ no teste após ___ s de treino. A curva de validação ___ [acompanhou / descolou da] de treino, sinal de ___. Como a CNN olha a imagem inteira, há o risco de ela aprender o *contexto* (mãos, obra, neve) em vez do objeto; isso explicaria ___.

## 9. Comparação crítica das três abordagens

A tabela numérica abaixo é gerada a partir dos resultados medidos acima. A avaliação qualitativa (facilidade de uso e integração) é minha leitura do que foi necessário para colocar cada abordagem em pé.

In [ ]:
# Tabela comparativa final (critérios do enunciado: facilidade, precisão, tempo de treino, tempo de inferência).
def facilidade(nome):
    """Leitura qualitativa de facilidade de uso/integração de cada abordagem."""
    if "custom" in nome:
        return "Média: exige dataset rotulado com caixas, mas o pipeline train/val/detect da YOLOv5 é pronto"
    if "padrão" in nome:
        return "Alta: nenhum treino, nenhum rótulo; só funciona para classes que o COCO conhece"
    return "Alta para rotular (só o nome da classe), baixa para generalizar; não localiza o objeto"

comparacao = pd.DataFrame([dict(abordagem=k, facilidade=facilidade(k),
                                acerto_por_imagem_teste=v.get("acerto_imagem"), mAP50_teste=v.get("mAP50_teste"),
                                treino_min=(v.get("treino_s") or 0) / 60, inferencia_ms_por_imagem=v.get("inferencia_ms_detect"))
                           for k, v in METRICAS.items()]).set_index("abordagem")
display(comparacao.round(3))
comparacao.to_csv(RESULTADOS / "comparacao_abordagens.csv")

# Visual: as 8 imagens de teste com o acerto de cada abordagem
painel = pd.concat(ACERTOS.values()).pivot(index="imagem", columns="modelo", values="acertou")
display(pd.DataFrame(np.where(painel.values, "✅", "❌"), index=painel.index, columns=painel.columns))

> ✍️ **[PREENCHER] Avaliação crítica**
>
> **Facilidade de uso/integração.** ___ [ex.: a YOLO padrão é a mais simples de integrar, mas só serve quando o objeto está no vocabulário do COCO; a customizada exigiu ~1h de rotulação para 80 imagens; a CNN dispensa caixas mas não resolve o problema do cliente, que precisa saber *onde* está o EPI].
>
> **Precisão.** ___ [ex.: no acerto por imagem, a customizada fez ___, a padrão ___ (acertou só celulares) e a CNN ___. A padrão falhou 100% em capacete, como esperado].
>
> **Tempo de treinamento/customização.** ___ [ex.: YOLO padrão: zero; CNN: ___ s; YOLO custom: ___ min (30 ép.) e ___ min (60 ép.) em T4].
>
> **Tempo de inferência.** ___ [ex.: as três YOLO têm o mesmo custo (~___ ms/imagem), pois a arquitetura é a mesma; a CNN leva ___ ms, mas faz menos (não localiza)].
>
> **Quando usar cada uma.** YOLO padrão quando o objeto está no COCO e não há tempo para rotular; YOLO customizada quando o objeto é específico do cliente ou é preciso localizar; CNN do zero quando a tarefa é só classificar, o volume de dados é grande e o objeto domina a imagem (ou como baseline didático).

## 10. Conclusão geral e recomendação ao cliente

> ✍️ **[PREENCHER]**
>
> Para o caso de uso da FarmTech (controle de acesso com verificação de EPI), a recomendação é ___ [ex.: a YOLOv5 customizada com ___ épocas], porque ___ [localiza os dois objetos, roda em ~___ ms por imagem e foi a única que reconheceu capacete]. O investimento necessário foi ___ [80 imagens rotuladas em ~1 h e minutos de GPU], e o próximo passo para produção seria ___ [coletar imagens da câmera real da portaria, incluir cenas com os dois objetos, ampliar o conjunto de validação e definir o limiar de confiança junto ao cliente].
>
> O que eu faria diferente: ___.

## Apêndice: reprodutibilidade do dataset

O `manifesto.csv` registra, para cada uma das 80 imagens, o split e o ID originais no Open Images. A célula abaixo (desligada por padrão) baixa exatamente as mesmas imagens com o `downloader.py` oficial do Open Images, a partir desses IDs. Os critérios do filtro automático estão descritos na seção 2.

In [ ]:
REPRODUZIR_DOWNLOAD = False   # True para baixar novamente as 80 imagens originais a partir dos IDs do manifesto

if REPRODUZIR_DOWNLOAD:
    !pip -q install boto3 botocore tqdm
    !wget -q -O /content/downloader.py https://raw.githubusercontent.com/openimages/dataset/master/downloader.py
    lista = Path("/content/lista_oi.txt")
    lista.write_text("\n".join(f"{s}/{i}" for s, i in zip(manifesto.openimages_split, manifesto.openimages_id)) + "\n")
    !python /content/downloader.py /content/lista_oi.txt --download_folder=/content/oi_original --num_processes=8
    print("imagens baixadas:", len(list(Path("/content/oi_original").glob("*.jpg"))))

## Referências

- Jocher, G. et al. **YOLOv5** (Ultralytics). https://github.com/ultralytics/yolov5
- Kuznetsova, A. et al. **The Open Images Dataset V4/V7**. IJCV 2020. https://storage.googleapis.com/openimages/web/index.html
- Lin, T.-Y. et al. **Microsoft COCO: Common Objects in Context**. ECCV 2014.
- Skalski, P. **Make Sense**: ferramenta de rotulação de imagens. https://www.makesense.ai
- Chollet, F. et al. **Keras** / TensorFlow: `image_dataset_from_directory`, camadas de pré-processamento.
- FIAP. Material da disciplina de Redes Neurais (capítulos de YOLO e CNN), Fase 6.